# COMP34812 NLI — Demo Notebook

**Group:** n  
**Category:** B — Deep Learning (non-transformer)  
**Model:** ResESIM + 1477d input embeddings

| Component | Dim | Reference |
|---|---|---|
| GloVe | 300 | Pennington et al., 2014 |
| ELMo | 1024 | Peters et al., 2018 |
| CharCNN | 100 | Kim et al., 2016 |
| POS embeddings | 50 | Universal Dependencies |
| Negation flags | 3 | spaCy dep parse + boundary |
| **Total** | **1477** | |

## Instructions

1. Upload `test.csv`, `meta.pt`, `best_model.pt` to `/content/`
2. Run all cells top to bottom
3. Predictions saved to `Group_n_B.csv`

**Input CSV:** must have `premise` and `hypothesis` columns  
**OneDrive links for model files:** see README.md

```
test.csv
    ↓  ELMo pre-computation  (~5 mins, A100)
    ↓  GloVe + CharCNN + POS + Negation
    ↓  inference_embeddings.npz  [N, 64, 1477]
    ↓  OracleNet (ResESIM)
    ↓  Group_n_B.csv
```

## 1. Environment Setup

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

# ── Download ELMo weights (Peters et al., 2018) ───────────────────────────────
import os
os.makedirs('/content/elmo_model', exist_ok=True)
!wget -q -O /content/elmo_model/options.json \
  https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_options.json
!wget -q -O /content/elmo_model/weights.hdf5 \
  https://s3-us-west-2.amazonaws.com/allennlp/models/elmo/2x4096_512_2048cnn_2xhighway/elmo_2x4096_512_2048cnn_2xhighway_weights.hdf5

# ── Python 3.10 venv for AllenNLP ELMo ───────────────────────────────────────
!pip install -q virtualenv
!virtualenv -p /usr/bin/python3.10 /content/myenv -q
!source /content/myenv/bin/activate && \
  /content/myenv/bin/pip install -q allennlp allennlp-models && \
  /content/myenv/bin/pip install -q 'numpy<2.0' --force-reinstall && \
  /content/myenv/bin/pip install -q torch==1.13.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117 && \
  /content/myenv/bin/pip install -q \
    https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.3.0/en_core_web_sm-3.3.0-py3-none-any.whl

print('Environment ready.')

## 2. Clone Repo + Upload Files

In [ ]:
# ── Clone repo ────────────────────────────────────────────────────────────────
!git clone https://github.com/VP1606/NLU-CW.git /content/NLU-CW

import sys
sys.path.append('/content/NLU-CW')

print('Repo cloned.')

In [ ]:
# ── Verify required files ─────────────────────────────────────────────────────
# Upload these before continuing:
#   test.csv       — marker's input data
#   meta.pt        — vocab + GloVe (OneDrive link in README)
#   best_model.pt  — trained model weights (OneDrive link in README)

import os

required = {
    '/content/test.csv':      'Input test CSV  (premise, hypothesis columns)',
    '/content/meta.pt':       'Vocab + GloVe meta  (download from OneDrive)',
    '/content/best_model.pt': 'Trained model weights  (download from OneDrive)',
}

all_ok = True
for path, desc in required.items():
    exists = os.path.exists(path)
    size   = f'  ({os.path.getsize(path)/1e6:.1f} MB)' if exists else ''
    print(f'  {"✓" if exists else "✗  MISSING"}  {path}{size}')
    print(f'           {desc}')
    if not exists: all_ok = False

print()
print('All files present — continue.' if all_ok else 'Upload missing files before continuing.')

## 3. Pre-compute Embeddings

Uses `EmbeddingPrecomputer` from `precompute.py` — same class used during training.
Five steps run automatically:
1. Load meta (vocab, char2idx, pos2idx, GloVe)
2. Build embedding layers (CharCNN, POS, GloVe)
3. ELMo pre-computation via Python 3.10 venv (~5 mins on A100)
4. Full 1477d pre-computation (GloVe + ELMo + CharCNN + POS + Negation)
5. Save `inference_embeddings.npz`

In [ ]:
from precomputeClasses import EmbeddingPrecomputer

pc = EmbeddingPrecomputer(
    meta_path    = '/content/meta.pt',
    elmo_options = '/content/elmo_model/options.json',
    elmo_weights = '/content/elmo_model/weights.hdf5',
    elmo_venv    = '/content/myenv/bin/python',
)

pc.run(
    csv_path   = '/content/test.csv',
    output_npz = '/content/inference_embeddings.npz',
)